In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv("/Users/tharmmm/Documents/store_sale_project/store-sales-time-series-forecasting/train.csv")
df.head(20)

,id,date,store_nbr,family,sales,onpromotion
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0
1,1,2013-01-01,1,BABY CARE,0.0,0
2,2,2013-01-01,1,BEAUTY,0.0,0
3,3,2013-01-01,1,BEVERAGES,0.0,0
4,4,2013-01-01,1,BOOKS,0.0,0
5,5,2013-01-01,1,BREAD/BAKERY,0.0,0
6,6,2013-01-01,1,CELEBRATION,0.0,0
7,7,2013-01-01,1,CLEANING,0.0,0
8,8,2013-01-01,1,DAIRY,0.0,0
9,9,2013-01-01,1,DELI,0.0,0


Create Calendar Features

In [3]:
df["date"] = pd.to_datetime(df["date"])

df["day_of_week"] = df["date"].dt.dayofweek
df["month"] = df["date"].dt.month
df["year"] = df["date"].dt.year
df["week_of_year"] = df["date"].dt.isocalendar().week
df["is_weekend"] = df["day_of_week"].isin([5,6]).astype(int)

In [4]:
df.head()

,id,date,store_nbr,family,sales,onpromotion,day_of_week,month,year,week_of_year,is_weekend
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0,1,1,2013,1,0
1,1,2013-01-01,1,BABY CARE,0.0,0,1,1,2013,1,0
2,2,2013-01-01,1,BEAUTY,0.0,0,1,1,2013,1,0
3,3,2013-01-01,1,BEVERAGES,0.0,0,1,1,2013,1,0
4,4,2013-01-01,1,BOOKS,0.0,0,1,1,2013,1,0


In [5]:
df_holiday = pd.read_csv("/Users/tharmmm/Documents/store_sale_project/store-sales-time-series-forecasting/holidays_events.csv")
df_holiday.head(20)

,date,type,locale,locale_name,description,transferred
0,2012-03-02,Holiday,Local,Manta,Fundacion de Manta,False
1,2012-04-01,Holiday,Regional,Cotopaxi,Provincializacion de Cotopaxi,False
2,2012-04-12,Holiday,Local,Cuenca,Fundacion de Cuenca,False
3,2012-04-14,Holiday,Local,Libertad,Cantonizacion de Libertad,False
4,2012-04-21,Holiday,Local,Riobamba,Cantonizacion de Riobamba,False
5,2012-05-12,Holiday,Local,Puyo,Cantonizacion del Puyo,False
6,2012-06-23,Holiday,Local,Guaranda,Cantonizacion de Guaranda,False
7,2012-06-25,Holiday,Regional,Imbabura,Provincializacion de Imbabura,False
8,2012-06-25,Holiday,Local,Latacunga,Cantonizacion de Latacunga,False
9,2012-06-25,Holiday,Local,Machala,Fundacion de Machala,False


In [6]:
df_holiday['type'].unique()

array(['Holiday', 'Transfer', 'Additional', 'Bridge', 'Work Day', 'Event'],
      dtype=object)

In [7]:
df_holiday[df_holiday['transferred'] == True]['transferred'].count()

np.int64(12)

In [8]:
df_holiday[df_holiday['type'] == "Work Day"]

,date,type,locale,locale_name,description,transferred
42,2013-01-05,Work Day,National,Ecuador,Recupero puente Navidad,False
43,2013-01-12,Work Day,National,Ecuador,Recupero puente primer dia del ano,False
149,2014-12-20,Work Day,National,Ecuador,Recupero Puente Navidad,False
161,2015-01-10,Work Day,National,Ecuador,Recupero Puente Primer dia del ano,False
283,2016-11-12,Work Day,National,Ecuador,Recupero Puente Dia de Difuntos,False


#### Clean df_holiday we get row that transferr == True and type != work day out

In [9]:
df_holidays_clean = df_holiday[
    (df_holiday["transferred"] == False) &
    (df_holiday["type"] != "Work Day")
]
print(f"before clean :{df_holiday.shape}")
print(f"after clean :{df_holidays_clean.shape}")

before clean :(350, 6)
after clean :(333, 6)


### Split types to new feature

In [10]:
holiday_types = ["Holiday", "Transfer", "Bridge", "Additional"]

holiday_flags = (
    df_holidays_clean
    .assign(
        is_holiday=lambda x: x["type"].isin(holiday_types),
        is_earthquake=lambda x: x["description"].str.contains("Terremoto", na=False),
        is_other_event=lambda x: (x["type"]=="Event") & (~x["description"].str.contains("Terremoto", na=False))
    )
    .groupby("date")[["is_holiday","is_earthquake","is_other_event"]]
    .max()
    .reset_index()
)

In [11]:
holiday_flags

,date,is_holiday,is_earthquake,is_other_event
0,2012-03-02,True,False,False
1,2012-04-01,True,False,False
2,2012-04-12,True,False,False
3,2012-04-14,True,False,False
4,2012-04-21,True,False,False
...,...,...,...,...
291,2017-12-22,True,False,False
292,2017-12-23,True,False,False
293,2017-12-24,True,False,False
294,2017-12-25,True,False,False


In [12]:
df.shape

(3000888, 11)

In [13]:
holiday_flags["date"] = pd.to_datetime(holiday_flags["date"])
df = df.merge(holiday_flags, on="date", how="left")

In [14]:
df.shape

(3000888, 14)

In [15]:
df.head()

,id,date,store_nbr,family,sales,onpromotion,day_of_week,month,year,week_of_year,is_weekend,is_holiday,is_earthquake,is_other_event
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0,1,1,2013,1,0,True,False,False
1,1,2013-01-01,1,BABY CARE,0.0,0,1,1,2013,1,0,True,False,False
2,2,2013-01-01,1,BEAUTY,0.0,0,1,1,2013,1,0,True,False,False
3,3,2013-01-01,1,BEVERAGES,0.0,0,1,1,2013,1,0,True,False,False
4,4,2013-01-01,1,BOOKS,0.0,0,1,1,2013,1,0,True,False,False


In [16]:
df[["is_holiday","is_earthquake","is_other_event"]] = (
    df[["is_holiday","is_earthquake","is_other_event"]]
    .fillna(0)
    .astype(int)
)

In [17]:
df.head()

,id,date,store_nbr,family,sales,onpromotion,day_of_week,month,year,week_of_year,is_weekend,is_holiday,is_earthquake,is_other_event
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0,1,1,2013,1,0,1,0,0
1,1,2013-01-01,1,BABY CARE,0.0,0,1,1,2013,1,0,1,0,0
2,2,2013-01-01,1,BEAUTY,0.0,0,1,1,2013,1,0,1,0,0
3,3,2013-01-01,1,BEVERAGES,0.0,0,1,1,2013,1,0,1,0,0
4,4,2013-01-01,1,BOOKS,0.0,0,1,1,2013,1,0,1,0,0


In [18]:
len(df[df['is_earthquake' ]== 1].groupby('date'))

31

In [19]:
df[["is_holiday","is_earthquake","is_other_event"]].sum()

is_holiday        338580
is_earthquake      55242
is_other_event     44550
dtype: int64

### promotion flag

In [20]:
df["is_promo"] = (df["onpromotion"] > 0).astype(int)

In [21]:
df['is_promo'].unique()

array([0, 1])

lag feature

In [22]:
df = df.sort_values(["store_nbr", "family", "date"])

# Lag by  7 day, 14 days, 28 days
for lag in [7, 14, 28]:
    df[f"sales_lag_{lag}"] = (
        df.groupby(["store_nbr", "family"])["sales"]
        .shift(lag)
    )

In [23]:
df.head(10)

,id,date,store_nbr,family,sales,onpromotion,day_of_week,month,year,week_of_year,is_weekend,is_holiday,is_earthquake,is_other_event,is_promo,sales_lag_7,sales_lag_14,sales_lag_28
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0,1,1,2013,1,0,1,0,0,0,NaN,NaN,NaN
1782,1782,2013-01-02,1,AUTOMOTIVE,2.0,0,2,1,2013,1,0,0,0,0,0,NaN,NaN,NaN
3564,3564,2013-01-03,1,AUTOMOTIVE,3.0,0,3,1,2013,1,0,0,0,0,0,NaN,NaN,NaN
5346,5346,2013-01-04,1,AUTOMOTIVE,3.0,0,4,1,2013,1,0,0,0,0,0,NaN,NaN,NaN
7128,7128,2013-01-05,1,AUTOMOTIVE,5.0,0,5,1,2013,1,1,0,0,0,0,NaN,NaN,NaN
8910,8910,2013-01-06,1,AUTOMOTIVE,2.0,0,6,1,2013,1,1,0,0,0,0,NaN,NaN,NaN
10692,10692,2013-01-07,1,AUTOMOTIVE,0.0,0,0,1,2013,2,0,0,0,0,0,NaN,NaN,NaN
12474,12474,2013-01-08,1,AUTOMOTIVE,2.0,0,1,1,2013,2,0,0,0,0,0,0.0,NaN,NaN
14256,14256,2013-01-09,1,AUTOMOTIVE,2.0,0,2,1,2013,2,0,0,0,0,0,2.0,NaN,NaN
16038,16038,2013-01-10,1,AUTOMOTIVE,2.0,0,3,1,2013,2,0,0,0,0,0,3.0,NaN,NaN


Rolling mean

In [24]:
df = df.sort_values(["store_nbr", "family", "date"])

for window in [7, 14, 28]:
    df[f"sales_rolling_mean_{window}"] = (
        df.groupby(["store_nbr", "family"])["sales"]
        .transform(lambda x: x.shift(1).rolling(window).mean())
    )

In [25]:
df.head()

,id,date,store_nbr,family,sales,onpromotion,day_of_week,month,year,week_of_year,...,is_holiday,is_earthquake,is_other_event,is_promo,sales_lag_7,sales_lag_14,sales_lag_28,sales_rolling_mean_7,sales_rolling_mean_14,sales_rolling_mean_28
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0,1,1,2013,1,...,1,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
1782,1782,2013-01-02,1,AUTOMOTIVE,2.0,0,2,1,2013,1,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
3564,3564,2013-01-03,1,AUTOMOTIVE,3.0,0,3,1,2013,1,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
5346,5346,2013-01-04,1,AUTOMOTIVE,3.0,0,4,1,2013,1,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
7128,7128,2013-01-05,1,AUTOMOTIVE,5.0,0,5,1,2013,1,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN


In [26]:
df.columns

Index(['id', 'date', 'store_nbr', 'family', 'sales', 'onpromotion',
       'day_of_week', 'month', 'year', 'week_of_year', 'is_weekend',
       'is_holiday', 'is_earthquake', 'is_other_event', 'is_promo',
       'sales_lag_7', 'sales_lag_14', 'sales_lag_28', 'sales_rolling_mean_7',
       'sales_rolling_mean_14', 'sales_rolling_mean_28'],
      dtype='object')

add Rolling Standard Deviation feature

In [27]:
df["sales_rolling_std_7"] = (
    df.groupby(["store_nbr","family"])["sales"]
    .transform(lambda x: x.shift(1).rolling(7).std())
)

In [28]:
df.head()

,id,date,store_nbr,family,sales,onpromotion,day_of_week,month,year,week_of_year,...,is_earthquake,is_other_event,is_promo,sales_lag_7,sales_lag_14,sales_lag_28,sales_rolling_mean_7,sales_rolling_mean_14,sales_rolling_mean_28,sales_rolling_std_7
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0,1,1,2013,1,...,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1782,1782,2013-01-02,1,AUTOMOTIVE,2.0,0,2,1,2013,1,...,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3564,3564,2013-01-03,1,AUTOMOTIVE,3.0,0,3,1,2013,1,...,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5346,5346,2013-01-04,1,AUTOMOTIVE,3.0,0,4,1,2013,1,...,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7128,7128,2013-01-05,1,AUTOMOTIVE,5.0,0,5,1,2013,1,...,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [29]:
df["promo_lag_7"] = (
    df.groupby(["store_nbr","family"])["onpromotion"]
    .shift(7)
)

In [30]:
df.head()

,id,date,store_nbr,family,sales,onpromotion,day_of_week,month,year,week_of_year,...,is_other_event,is_promo,sales_lag_7,sales_lag_14,sales_lag_28,sales_rolling_mean_7,sales_rolling_mean_14,sales_rolling_mean_28,sales_rolling_std_7,promo_lag_7
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0,1,1,2013,1,...,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1782,1782,2013-01-02,1,AUTOMOTIVE,2.0,0,2,1,2013,1,...,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3564,3564,2013-01-03,1,AUTOMOTIVE,3.0,0,3,1,2013,1,...,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5346,5346,2013-01-04,1,AUTOMOTIVE,3.0,0,4,1,2013,1,...,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7128,7128,2013-01-05,1,AUTOMOTIVE,5.0,0,5,1,2013,1,...,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Encoder Family column

In [31]:
df["family"] = df["family"].astype("category")
df.head()

,id,date,store_nbr,family,sales,onpromotion,day_of_week,month,year,week_of_year,...,is_other_event,is_promo,sales_lag_7,sales_lag_14,sales_lag_28,sales_rolling_mean_7,sales_rolling_mean_14,sales_rolling_mean_28,sales_rolling_std_7,promo_lag_7
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0,1,1,2013,1,...,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1782,1782,2013-01-02,1,AUTOMOTIVE,2.0,0,2,1,2013,1,...,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3564,3564,2013-01-03,1,AUTOMOTIVE,3.0,0,3,1,2013,1,...,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5346,5346,2013-01-04,1,AUTOMOTIVE,3.0,0,4,1,2013,1,...,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7128,7128,2013-01-05,1,AUTOMOTIVE,5.0,0,5,1,2013,1,...,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Add stores feature

In [32]:
df_stores = pd.read_csv("/Users/tharmmm/Documents/store_sale_project/store-sales-time-series-forecasting/stores.csv")
df = df.merge(df_stores, on="store_nbr", how="left")
df.head()

,id,date,store_nbr,family,sales,onpromotion,day_of_week,month,year,week_of_year,...,sales_lag_28,sales_rolling_mean_7,sales_rolling_mean_14,sales_rolling_mean_28,sales_rolling_std_7,promo_lag_7,city,state,type,cluster
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0,1,1,2013,1,...,NaN,NaN,NaN,NaN,NaN,NaN,Quito,Pichincha,D,13
1,1782,2013-01-02,1,AUTOMOTIVE,2.0,0,2,1,2013,1,...,NaN,NaN,NaN,NaN,NaN,NaN,Quito,Pichincha,D,13
2,3564,2013-01-03,1,AUTOMOTIVE,3.0,0,3,1,2013,1,...,NaN,NaN,NaN,NaN,NaN,NaN,Quito,Pichincha,D,13
3,5346,2013-01-04,1,AUTOMOTIVE,3.0,0,4,1,2013,1,...,NaN,NaN,NaN,NaN,NaN,NaN,Quito,Pichincha,D,13
4,7128,2013-01-05,1,AUTOMOTIVE,5.0,0,5,1,2013,1,...,NaN,NaN,NaN,NaN,NaN,NaN,Quito,Pichincha,D,13


Add oil feature

In [33]:
df_oil = pd.read_csv("/Users/tharmmm/Documents/store_sale_project/store-sales-time-series-forecasting/oil.csv")
df_oil["date"] = pd.to_datetime(df_oil["date"])
df = df.merge(df_oil, on="date", how="left")


In [34]:
df.head()

,id,date,store_nbr,family,sales,onpromotion,day_of_week,month,year,week_of_year,...,sales_rolling_mean_7,sales_rolling_mean_14,sales_rolling_mean_28,sales_rolling_std_7,promo_lag_7,city,state,type,cluster,dcoilwtico
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0,1,1,2013,1,...,NaN,NaN,NaN,NaN,NaN,Quito,Pichincha,D,13,NaN
1,1782,2013-01-02,1,AUTOMOTIVE,2.0,0,2,1,2013,1,...,NaN,NaN,NaN,NaN,NaN,Quito,Pichincha,D,13,93.14
2,3564,2013-01-03,1,AUTOMOTIVE,3.0,0,3,1,2013,1,...,NaN,NaN,NaN,NaN,NaN,Quito,Pichincha,D,13,92.97
3,5346,2013-01-04,1,AUTOMOTIVE,3.0,0,4,1,2013,1,...,NaN,NaN,NaN,NaN,NaN,Quito,Pichincha,D,13,93.12
4,7128,2013-01-05,1,AUTOMOTIVE,5.0,0,5,1,2013,1,...,NaN,NaN,NaN,NaN,NaN,Quito,Pichincha,D,13,NaN


In [35]:
print("The number of NaN in oil column :", df["dcoilwtico"].isnull().sum()) 


The number of NaN in oil column : 928422


In [36]:
df["dcoilwtico"] = df["dcoilwtico"].ffill().bfill()
print("The number of NaN in oil column after fill Nan :", df["dcoilwtico"].isnull().sum()) 


The number of NaN in oil column after fill Nan : 0


In [37]:
df.columns

Index(['id', 'date', 'store_nbr', 'family', 'sales', 'onpromotion',
       'day_of_week', 'month', 'year', 'week_of_year', 'is_weekend',
       'is_holiday', 'is_earthquake', 'is_other_event', 'is_promo',
       'sales_lag_7', 'sales_lag_14', 'sales_lag_28', 'sales_rolling_mean_7',
       'sales_rolling_mean_14', 'sales_rolling_mean_28', 'sales_rolling_std_7',
       'promo_lag_7', 'city', 'state', 'type', 'cluster', 'dcoilwtico'],
      dtype='object')

Add feature day of month and is_month_start and is_month_end

In [38]:
df["day_of_month"] = df["date"].dt.day

In [39]:
df["is_month_start"] = df["date"].dt.is_month_start.astype(int)
df["is_month_end"] = df["date"].dt.is_month_end.astype(int)


In [40]:
df.head()

,id,date,store_nbr,family,sales,onpromotion,day_of_week,month,year,week_of_year,...,sales_rolling_std_7,promo_lag_7,city,state,type,cluster,dcoilwtico,day_of_month,is_month_start,is_month_end
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0,1,1,2013,1,...,NaN,NaN,Quito,Pichincha,D,13,93.14,1,1,0
1,1782,2013-01-02,1,AUTOMOTIVE,2.0,0,2,1,2013,1,...,NaN,NaN,Quito,Pichincha,D,13,93.14,2,0,0
2,3564,2013-01-03,1,AUTOMOTIVE,3.0,0,3,1,2013,1,...,NaN,NaN,Quito,Pichincha,D,13,92.97,3,0,0
3,5346,2013-01-04,1,AUTOMOTIVE,3.0,0,4,1,2013,1,...,NaN,NaN,Quito,Pichincha,D,13,93.12,4,0,0
4,7128,2013-01-05,1,AUTOMOTIVE,5.0,0,5,1,2013,1,...,NaN,NaN,Quito,Pichincha,D,13,93.12,5,0,0


Encode city and state

In [41]:
df["city"] = df["city"].astype("category")
df["state"] = df["state"].astype("category")
df["type"] = df["type"].astype("category")

Save training data

In [42]:
feature_cols = [col for col in df.columns if col not in ["id", "sales"]]

X = df[feature_cols]
y = df["sales"]


In [43]:
df[feature_cols + ["sales"]].to_parquet("train_featured.parquet", index=False)

try to load training data

In [44]:
df_train = pd.read_parquet("/Users/tharmmm/Documents/store_sale_project/store-sales-time-series-forecasting/train_featured.parquet")
df_train.head()

FileNotFoundError: [Errno 2] No such file or directory: '/Users/tharmmm/Documents/store_sale_project/store-sales-time-series-forecasting/train_featured.parquet'

In [ ]:
df_train.columns

Index(['store_nbr', 'family', 'onpromotion', 'day_of_week', 'month', 'year',
       'week_of_year', 'is_weekend', 'is_holiday', 'is_earthquake',
       'is_other_event', 'is_promo', 'sales_lag_1', 'sales_lag_7',
       'sales_lag_28', 'sales_rolling_mean_7', 'sales_rolling_mean_14',
       'sales_rolling_mean_28', 'sales_rolling_std_7', 'promo_lag_7', 'city',
       'state', 'type', 'cluster', 'dcoilwtico', 'transactions',
       'transactions_lag_7', 'day_of_month', 'is_month_start', 'is_month_end',
       'sales'],
      dtype='object')

In [ ]:
len(df_train.columns)

31